# Лаб№1. Работа с API Романов Михаил ПМИ-3

1. Разобраться по документации GitHub API, как вывести список репозиториев конкретного пользователя, сохранить JSON-вывод в файл .json.
2. Изучить список открытых API на ProgrammableWeb, найти API, требующее авторизацию (любого типа), выполнить запрос, пройдя авторизацию, и записать ответ сервера в файл.

## Задание 1. Список репозиториев пользователя GitHub

По [документации GitHub REST API v3](https://docs.github.com/en/rest/repos/repos#list-repositories-for-a-user) список публичных репозиториев пользователя получается запросом:

GET https://api.github.com/users/{username}/repos
Параметры запроса:
- per_page — число записей на странице (максимум 100);
- page — номер страницы (пагинация);
- sort — created | updated | pushed | full_name;
- type — owner | member | all.


In [2]:
USERNAME = "PINNOS"
API_URL = f"https://api.github.com/users/{USERNAME}/repos"

repos = []
page = 1
MAX_PAGES = 20  # жёсткий предел, чтобы исключить бесконечный цикл

while page <= MAX_PAGES:
    resp = get_with_retry(
        API_URL, params={"per_page": 100, "page": page, "sort": "updated"}
    )
    batch = resp.json()
    if not batch:
        break
    repos.extend(batch)
    if len(batch) < 100:
        break
    page += 1

# Сохранение JSON-вывода в файл
with open("repos.json", "w", encoding="utf-8") as f:
    json.dump(repos, f, indent=2, ensure_ascii=False)

print(f"Всего репозиториев: {len(repos)}, пройдено страниц: {page}")
print("\nПервые 5 репозиториев (name, stars, language, updated_at):")
for item in repos[:5]:
    print(
        f"  - {item['full_name']}: stars={item['stargazers_count']}, "
        f"{item['language']}, updated={item['updated_at']}"
    )

Всего репозиториев: 7, пройдено страниц: 1

Первые 5 репозиториев (name, stars, language, updated_at):
  - PINNOS/parsing-2026: stars=0, None, updated=2026-09-09T09:22:46Z
  - PINNOS/ckusters_2trim: stars=0, Jupyter Notebook, updated=2026-04-22T13:39:24Z
  - PINNOS/neural-network_2trim: stars=0, Jupyter Notebook, updated=2026-04-21T14:48:55Z
  - PINNOS/regres_2trim_2026: stars=0, Jupyter Notebook, updated=2026-04-07T15:40:15Z
  - PINNOS/FileTimestampChanger: stars=0, TeX, updated=2026-03-30T12:08:50Z


API вернул список публичных репозиториев пользователя PINNOS, все записи выгружены с учётом пагинации (при их числе больше 100 запросились бы вторая страница и далее, пока страница не станет неполной). Ответ сервера сохранён в 1lab/repos.json в исходном виде.

## Задание 2. API с авторизацией

Выбрал OpenWeatherMap API (`https://openweathermap.org/api`)  тип авторизации API Key: ключ передаётся в query-параметре appid вместе с запросом.
Запрос: `GET https://api.openweathermap.org/data/2.5/weather?q={город}&units=metric` текущая погода по городу.
Ключ хранится в файле `.env` в корне репозитория (строка `DEMO_API_KEY=...`) и читается через `os.getenv()`; сам `.env` в контроль версий не попадает. Ответ сервера записывается в файл `weather_response.json`.

In [3]:
API_KEY = os.getenv("DEMO_API_KEY") or os.getenv("OPENWEATHERMAP_KEY")
if not API_KEY:
    raise SystemExit(
        "API-ключ не найден: добавьте строку DEMO_API_KEY=<ключ> в файл .env"
    )

CITY = "Perm"
URL = "https://api.openweathermap.org/data/2.5/weather"

resp = get_with_retry(
    URL,
    params={"q": CITY, "appid": API_KEY, "units": "metric", "lang": "ru"},
    retries=3,
    timeout=15,
)
weather = resp.json()

# Ответ сервера записываем в файл
with open("weather_response.json", "w", encoding="utf-8") as f:
    json.dump(weather, f, ensure_ascii=False, indent=4)

print(f"HTTP {resp.status_code}, авторизация по API Key пройдена")
print(
    f"{weather['name']}: {weather['weather'][0]['description']}, "
    f"{weather['main']['temp']} °C, влажность {weather['main']['humidity']}%"
)

HTTP 200, авторизация по API Key пройдена
Пермь: переменная облачность, 2.7 °C, влажность 65%


сервер ответил кодом 200 и текущей погодой по городу Пермь (описание состояния, температура, влажность). Ответ сервера целиком сохранён в `1lab/weather_response.json` файл содержит все поля ответа API.